미션: 모두마켓 "분기 성과 요약 보고서" 만들기
아래 단계를 따라 하나의 분석 노트북을 완성하고 GitHub에 제출하세요.

데이터 준비: 이 노트북의 종합 실습 데이터 생성 셀(np.random.seed(7) 셀)을 그대로 가져와 orders·customers·products를 만드세요.
키 검증 (필수): 병합 전 세 검증(중복·미매칭·관계)을 수행하고, 결과를 마크다운으로 기록하세요. ("검증 결과 ___ 이므로 m:1 병합이 안전하다.")
병합: validate를 걸어 세 테이블을 하나로 합치세요.
집계: 다음 두 요약표를 만드세요.
▸
지역 × 회원등급 매출 교차표 (pivot_table)
▸
카테고리별 KPI(매출·건수·객단가) 요약표 (groupby + agg)
특성 추가: transform으로 "각 주문이 그 지역 매출에서 차지하는 비중" 컬럼을 추가하세요.
보고서: 종합 실습의 보고서 양식을 채워, 발견 3가지 + 제언 1가지를 작성하세요.

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# ═════════════════════════════════════════════
# 모두마켓 분기 성과 요약 보고서
# ═════════════════════════════════════════════

# ── 1. 데이터 준비 ──────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})

orders, customers, products = ordr, cust, prod

display(Markdown("# 모두마켓 분기 성과 요약 보고서"))
display(Markdown("## 1. 데이터 준비"))
print("orders:", orders.shape, "| customers:", customers.shape, "| products:", products.shape)

# ── 2. 키 검증 ──────────────────────────────
dup_c = customers["customer_id"].duplicated().sum()
dup_p = products["product_id"].duplicated().sum()
miss_c = (~orders["customer_id"].isin(customers["customer_id"])).sum()
miss_p = (~orders["product_id"].isin(products["product_id"])).sum()

display(Markdown(f"""
## 2. 키 검증

| 검증 항목 | 결과 |
|---|---|
| customers 키 중복 | {dup_c}건 |
| products 키 중복 | {dup_p}건 |
| 미매칭 customer_id | {miss_c}건 |
| 미매칭 product_id | {miss_p}건 |
| 관계 | orders(다) : customers(1), orders(다) : products(1) → validate="m:1"로 확인 |

**검증 결과 룩업 표의 키가 모두 유일하고 주문의 모든 키가 매칭되므로 m:1 병합이 안전하다.**
"""))

# ── 3. 병합 ─────────────────────────────────
df = (
    orders
    .merge(customers, on="customer_id", how="left", validate="m:1")
    .merge(products, on="product_id", how="left", validate="m:1")
)

display(Markdown(f"""
## 3. 병합
병합 후 shape: {df.shape} / 행 수 보존: {len(df) == len(orders)} / 결측치: {df.isna().sum().sum()}개
"""))
display(df.head(3))

# ── 4. 집계 ─────────────────────────────────
region_member = pd.pivot_table(
    df, index="region", columns="membership", values="amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="합계",
).round(0)

member_count = pd.crosstab(
    customers["region"], customers["membership"],
    margins=True, margins_name="합계",
)

category_kpi = (
    df.groupby("category")
    .agg(
        매출=("amount", "sum"),
        건수=("order_id", "count"),
        객단가=("amount", "mean"),
    )
    .round(0)
    .sort_values("매출", ascending=False)
)
category_kpi["매출비중(%)"] = (category_kpi["매출"] / category_kpi["매출"].sum() * 100).round(1)
category_kpi["상품수"] = products.groupby("category")["product_id"].count()

display(Markdown("## 4. 집계\n### 4-1. 지역 × 회원등급 매출 교차표"))
display(region_member)
display(Markdown("(참고) 지역 × 회원등급 고객 수"))
display(member_count)
display(Markdown("### 4-2. 카테고리별 KPI 요약표"))
display(category_kpi)

# ── 5. 특성 추가 ────────────────────────────
df["region_total"] = df.groupby("region")["amount"].transform("sum")
df["share_in_region(%)"] = (df["amount"] / df["region_total"] * 100).round(3)

check = (df.groupby("region")["amount"].sum()
         / df.groupby("region")["region_total"].first() * 100).round(1)

display(Markdown(f"""
## 5. 특성 추가 — 지역 매출 내 주문 비중
transform 후 행 수: {len(df)} (줄어들지 않음) / 지역별 비중 합계 검산: {", ".join(f"{k} {v}%" for k, v in check.items())}
"""))
display(df[["order_id", "region", "amount", "region_total", "share_in_region(%)"]].head())

# ── 6. 보고서 ───────────────────────────────
region_kpi = df.groupby("region").agg(
    매출=("amount", "sum"),
    고객수=("customer_id", "nunique"),
    객단가=("amount", "mean"),
)
region_kpi["고객당매출"] = region_kpi["매출"] / region_kpi["고객수"]
region_kpi = region_kpi.round(0).sort_values("매출", ascending=False)

member_kpi = df.groupby("membership").agg(
    매출=("amount", "sum"),
    고객수=("customer_id", "nunique"),
    객단가=("amount", "mean"),
)
member_kpi["고객당매출"] = member_kpi["매출"] / member_kpi["고객수"]
member_kpi = member_kpi.round(0)

total = df["amount"].sum()
aov = df["amount"].mean()

display(Markdown("## 6. 보고서\n### 보고서 근거 표 — 지역별 / 등급별 KPI"))
display(region_kpi)
display(member_kpi)

display(Markdown(f"""
### 모두마켓 분기 성과 요약

**개요**
분석 기간(2025년 1~5월) 동안 모두마켓은 주문 {len(df):,}건, 총매출 약 {total/1e4:,.0f}만 원을 기록했으며 전체 객단가는 약 {aov/1e4:.1f}만 원이다.

**발견 1. 패션이 전체 매출의 42%를 차지하는 핵심 카테고리다.**
패션 매출은 약 3,832만 원으로 2위 식품(약 2,240만 원)의 1.7배다. 주문 건수(512건)와 객단가(약 7.5만 원) 모두 4개 카테고리 중 1위로, 많이 팔리면서 한 번에 쓰는 금액도 크다. 반면 뷰티는 매출 비중 11%, 주문 192건으로 가장 작은데, 취급 상품이 4종뿐이라 선택지가 적은 것이 원인으로 보인다.

**발견 2. 인천은 매출 1위지만, 이는 고객 수가 많기 때문이다.**
인천 매출(약 2,758만 원)은 4개 지역 중 가장 크지만 고객 수도 61명으로 가장 많다. 고객 1인당 매출로 보면 경기(약 47.0만 원)가 1위이고, 인천(약 45.2만 원)은 부산(약 45.7만 원)보다도 낮은 3위다. 경기는 객단가도 약 6.6만 원으로 가장 높아, 고객 한 명의 가치가 가장 큰 지역이다.

**발견 3. 인천은 vip 고객이 1명뿐이라 등급 구성이 한쪽으로 쏠려 있다.**
지역 × 회원등급 교차표에서 인천 vip 매출은 53만 원으로 다른 지역 vip(232만~333만 원)보다 크게 낮다. 고객 구성을 보면 인천 vip는 1명뿐인 반면 premium은 22명으로 전 지역 중 가장 많고, 인천 premium 매출(약 968만 원)도 전 지역 premium 중 1위다. 즉 인천에는 vip로 올라갈 수 있는 premium 고객층이 두텁다.

**제언. 인천 premium 고객 대상 vip 승급 프로모션을 실시한다.**
인천은 고객 수가 가장 많고 premium 고객이 22명으로 두터운데도 vip 전환은 거의 없다. 등급별 고객당 매출은 vip(약 47.5만 원)가 basic(약 44.7만 원)보다 높으므로, 인천 premium 고객 중 구매 빈도가 높은 고객을 골라 승급 혜택(무료 배송, 패션 카테고리 할인 등)을 제안하면 핵심 지역의 고객 가치를 높일 수 있다. 실행 후에는 인천 vip 고객 수와 고객당 매출을 다음 분기 KPI로 추적한다.

**한계**
분석 기간이 5월 30일까지라 5월 데이터가 하루 부족하며, 실습용으로 생성된 데이터라 가격(4종)과 수량(1~3개)이 단순하다. vip 고객이 18명뿐이라 등급별 비교는 표본이 작다는 점을 감안해야 한다.
"""))

# 모두마켓 분기 성과 요약 보고서

## 1. 데이터 준비

orders: (1500, 6) | customers: (200, 3) | products: (30, 3)



## 2. 키 검증

| 검증 항목 | 결과 |
|---|---|
| customers 키 중복 | 0건 |
| products 키 중복 | 0건 |
| 미매칭 customer_id | 0건 |
| 미매칭 product_id | 0건 |
| 관계 | orders(다) : customers(1), orders(다) : products(1) → validate="m:1"로 확인 |

**검증 결과 룩업 표의 키가 모두 유일하고 주문의 모든 키가 매칭되므로 m:1 병합이 안전하다.**



## 3. 병합
병합 후 shape: (1500, 10) / 행 수 보존: True / 결측치: 0개


...

,order_id,region,amount,region_total,share_in_region(%)
0,O00001,서울,120000.0,23480000.0,0.511
1,O00002,서울,25000.0,23480000.0,0.106
2,O00003,인천,36000.0,27577000.0,0.131
3,O00004,경기,120000.0,21145000.0,0.568
4,O00005,경기,120000.0,21145000.0,0.568


## 6. 보고서
### 보고서 근거 표 — 지역별 / 등급별 KPI

,매출,고객수,객단가,고객당매출
region,,,,
인천,27577000.0,61,59433.0,452082.0
서울,23480000.0,53,59144.0,443019.0
경기,21145000.0,45,66078.0,469889.0
부산,18754000.0,41,58790.0,457415.0


,매출,고객수,객단가,고객당매출
membership,,,,
basic,55411000.0,124,59518.0,446863.0
premium,27002000.0,58,61090.0,465552.0
vip,8543000.0,18,67268.0,474611.0



### 모두마켓 분기 성과 요약

**개요**
분석 기간(2025년 1~5월) 동안 모두마켓은 주문 1,500건, 총매출 약 9,096만 원을 기록했으며 전체 객단가는 약 6.1만 원이다.

**발견 1. 패션이 전체 매출의 42%를 차지하는 핵심 카테고리다.**
패션 매출은 약 3,832만 원으로 2위 식품(약 2,240만 원)의 1.7배다. 주문 건수(512건)와 객단가(약 7.5만 원) 모두 4개 카테고리 중 1위로, 많이 팔리면서 한 번에 쓰는 금액도 크다. 반면 뷰티는 매출 비중 11%, 주문 192건으로 가장 작은데, 취급 상품이 4종뿐이라 선택지가 적은 것이 원인으로 보인다.

**발견 2. 인천은 매출 1위지만, 이는 고객 수가 많기 때문이다.**
인천 매출(약 2,758만 원)은 4개 지역 중 가장 크지만 고객 수도 61명으로 가장 많다. 고객 1인당 매출로 보면 경기(약 47.0만 원)가 1위이고, 인천(약 45.2만 원)은 부산(약 45.7만 원)보다도 낮은 3위다. 경기는 객단가도 약 6.6만 원으로 가장 높아, 고객 한 명의 가치가 가장 큰 지역이다.

**발견 3. 인천은 vip 고객이 1명뿐이라 등급 구성이 한쪽으로 쏠려 있다.**
지역 × 회원등급 교차표에서 인천 vip 매출은 53만 원으로 다른 지역 vip(232만~333만 원)보다 크게 낮다. 고객 구성을 보면 인천 vip는 1명뿐인 반면 premium은 22명으로 전 지역 중 가장 많고, 인천 premium 매출(약 968만 원)도 전 지역 premium 중 1위다. 즉 인천에는 vip로 올라갈 수 있는 premium 고객층이 두텁다.

**제언. 인천 premium 고객 대상 vip 승급 프로모션을 실시한다.**
인천은 고객 수가 가장 많고 premium 고객이 22명으로 두터운데도 vip 전환은 거의 없다. 등급별 고객당 매출은 vip(약 47.5만 원)가 basic(약 44.7만 원)보다 높으므로, 인천 premium 고객 중 구매 빈도가 높은 고객을 골라 승급 혜택(무료 배송, 패션 카테고리 할인 등)을 제안하면 핵심 지역의 고객 가치를 높일 수 있다. 실행 후에는 인천 vip 고객 수와 고객당 매출을 다음 분기 KPI로 추적한다.

**한계**
분석 기간이 5월 30일까지라 5월 데이터가 하루 부족하며, 실습용으로 생성된 데이터라 가격(4종)과 수량(1~3개)이 단순하다. vip 고객이 18명뿐이라 등급별 비교는 표본이 작다는 점을 감안해야 한다.
